In [ ]:
# ============================================================
# CÉLULA 1 — Instalação de Dependências
# ============================================================

!pip install apify-client networkx pandas matplotlib python-louvain --quiet
print("✅ Dependências instaladas com sucesso.")

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração da API Key do Apify e Parâmetros Globais
# ============================================================

from apify_client import ApifyClient

# Obtenha seu token em: https://console.apify.com/account/integrations
APIFY_TOKEN = "SEU_TOKEN_AQUI"

# Alternativa: usar Google Colab Secrets (descomente as linhas abaixo)
# from google.colab import userdata
# APIFY_TOKEN = userdata.get('APIFY_TOKEN')

# Parâmetros globais
HASHTAG = "inteligenciaartificial"
MAX_VIDEOS = 80
MAX_COMMENTS = 100

# Instancia o client da API
client = ApifyClient(APIFY_TOKEN)

print(f"✅ Cliente Apify configurado.")
print(f"   Hashtag: #{HASHTAG}")
print(f"   Máx. vídeos: {MAX_VIDEOS}")
print(f"   Máx. comentários por vídeo: {MAX_COMMENTS}")

In [ ]:
# ============================================================
# CÉLULA 3 — Coleta de Vídeos via Apify
# ============================================================

import json

print(f"🔄 Coletando vídeos com hashtag #{HASHTAG}...")

run_videos = client.actor("clockworks/tiktok-scraper").call(
    run_input={
        "hashtags": [HASHTAG],
        "resultsPerPage": MAX_VIDEOS,
        "maxRequestRetries": 5,
        "proxyConfiguration": {"useApifyProxy": True},
    }
)

videos_raw = []
for item in client.dataset(run_videos["defaultDatasetId"]).iterate_items():
    videos_raw.append(item)

print(f"✅ {len(videos_raw)} vídeos coletados.")

# Preview do primeiro item (primeiros 1500 caracteres)
if videos_raw:
    preview = json.dumps(videos_raw[0], indent=2, ensure_ascii=False)
    print("\n--- Preview do primeiro vídeo (1500 chars) ---")
    print(preview[:1500])

In [ ]:
# ============================================================
# CÉLULA 4 — Coleta de Comentários por Vídeo
# ============================================================

import json

# Extrai URLs dos vídeos coletados
video_urls = []
for v in videos_raw:
    url = v.get("webVideoUrl") or v.get("url") or ""
    if url:
        video_urls.append(url)

print(f"🔄 Coletando comentários de {len(video_urls)} vídeos...")

run_comments = client.actor("clockworks/tiktok-comments-scraper").call(
    run_input={
        "postURLs": video_urls,
        "commentsPerPost": MAX_COMMENTS,
    }
)

comments_raw = []
for item in client.dataset(run_comments["defaultDatasetId"]).iterate_items():
    comments_raw.append(item)

print(f"✅ {len(comments_raw)} comentários coletados.")

# Preview do primeiro comentário (800 caracteres)
if comments_raw:
    preview = json.dumps(comments_raw[0], indent=2, ensure_ascii=False)
    print("\n--- Preview do primeiro comentário (800 chars) ---")
    print(preview[:800])

In [ ]:
# ============================================================
# CÉLULA 5 — Criação do Banco SQLite e Ingestão dos Dados
# ============================================================

import sqlite3
import pandas as pd
import re

DB_PATH = "/content/tiktok_sna.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Criação das tabelas
cur.executescript("""
    DROP TABLE IF EXISTS videos;
    DROP TABLE IF EXISTS video_hashtags;
    DROP TABLE IF EXISTS comments;

    CREATE TABLE videos (
        video_id   TEXT PRIMARY KEY,
        author_id  TEXT,
        author_name TEXT,
        create_time INTEGER,
        description TEXT,
        likes      INTEGER DEFAULT 0,
        comments_cnt INTEGER DEFAULT 0,
        shares     INTEGER DEFAULT 0,
        views      INTEGER DEFAULT 0,
        url        TEXT
    );

    CREATE TABLE video_hashtags (
        video_id TEXT,
        hashtag  TEXT,
        PRIMARY KEY (video_id, hashtag)
    );

    CREATE TABLE comments (
        comment_id    TEXT PRIMARY KEY,
        video_id      TEXT,
        commenter_id  TEXT,
        commenter_name TEXT,
        text          TEXT,
        likes         INTEGER DEFAULT 0,
        created_at    INTEGER
    );
""")


def extract_hashtags(text):
    """Extrai hashtags de um texto usando regex."""
    if not text:
        return []
    return re.findall(r"#(\w+)", text)


# --- Ingestão de vídeos ---
for v in videos_raw:
    video_id = str(v.get("id") or v.get("videoId") or "")
    if not video_id:
        continue

    # Normaliza autor (variações do actor Apify)
    author_id = str(
        v.get("authorMeta", {}).get("id")
        or v.get("author", {}).get("id")
        or v.get("authorId")
        or ""
    )
    author_name = (
        v.get("authorMeta", {}).get("nickName")
        or v.get("authorMeta", {}).get("name")
        or v.get("author", {}).get("nickname")
        or v.get("author", {}).get("uniqueId")
        or v.get("authorName")
        or ""
    )

    # Normaliza estatísticas
    stats = v.get("stats", {})
    likes = int(v.get("diggCount") or stats.get("diggCount") or v.get("likes") or 0)
    comments_cnt = int(v.get("commentCount") or stats.get("commentCount") or v.get("comments") or 0)
    shares = int(v.get("shareCount") or stats.get("shareCount") or v.get("shares") or 0)
    views = int(v.get("playCount") or stats.get("playCount") or v.get("views") or 0)

    create_time = int(v.get("createTime") or v.get("created_at") or 0)
    description = v.get("text") or v.get("desc") or v.get("description") or ""
    url = v.get("webVideoUrl") or v.get("url") or ""

    cur.execute(
        "INSERT OR IGNORE INTO videos VALUES (?,?,?,?,?,?,?,?,?,?)",
        (video_id, author_id, author_name, create_time, description,
         likes, comments_cnt, shares, views, url),
    )

    # Insere hashtags extraídas da descrição
    for tag in extract_hashtags(description):
        cur.execute(
            "INSERT OR IGNORE INTO video_hashtags VALUES (?,?)",
            (video_id, tag.lower()),
        )

conn.commit()

# --- Ingestão de comentários ---
for c in comments_raw:
    comment_id = str(c.get("cid") or c.get("id") or c.get("commentId") or "")
    if not comment_id:
        continue

    # Normaliza vídeo de origem
    vid = str(
        c.get("videoId")
        or c.get("aweme_id")
        or c.get("videoWebUrl", "").split("/")[-1]
        or ""
    )

    # Normaliza dados do comentador
    user = c.get("user", {})
    commenter_id = str(user.get("uid") or c.get("userId") or user.get("id") or "")
    commenter_name = (
        user.get("nickname")
        or c.get("userName")
        or user.get("uniqueId")
        or ""
    )

    text = c.get("text") or c.get("comment") or ""
    likes = int(c.get("digg_count") or c.get("likes") or 0)
    created_at = int(c.get("create_time") or c.get("createdAt") or 0)

    cur.execute(
        "INSERT OR IGNORE INTO comments VALUES (?,?,?,?,?,?,?)",
        (comment_id, vid, commenter_id, commenter_name, text, likes, created_at),
    )

conn.commit()

# Contagem final
n_videos = cur.execute("SELECT COUNT(*) FROM videos").fetchone()[0]
n_hashtags = cur.execute("SELECT COUNT(*) FROM video_hashtags").fetchone()[0]
n_comments = cur.execute("SELECT COUNT(*) FROM comments").fetchone()[0]

print(f"✅ Banco SQLite criado em {DB_PATH}")
print(f"   videos:         {n_videos} registros")
print(f"   video_hashtags: {n_hashtags} registros")
print(f"   comments:       {n_comments} registros")

conn.close()

In [ ]:
# ============================================================
# CÉLULA 6 — Análise Exploratória dos Dados (EDA)
# ============================================================

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DB_PATH = "/content/tiktok_sna.db"
conn = sqlite3.connect(DB_PATH)

df_videos = pd.read_sql("SELECT * FROM videos", conn)
df_hashtags = pd.read_sql("SELECT * FROM video_hashtags", conn)
df_comments = pd.read_sql("SELECT * FROM comments", conn)

conn.close()

# Sumário geral
print("=" * 50)
print("SUMÁRIO DOS DADOS COLETADOS")
print("=" * 50)
print(f"Vídeos únicos:       {df_videos['video_id'].nunique()}")
print(f"Criadores únicos:    {df_videos['author_id'].nunique()}")
print(f"Hashtags únicas:     {df_hashtags['hashtag'].nunique()}")
print(f"Total de comentários:{df_comments.shape[0]}")
print(f"Comentadores únicos: {df_comments['commenter_id'].nunique()}")

# Top 15 hashtags
top_hashtags = df_hashtags["hashtag"].value_counts().head(15)
print("\n--- Top 15 Hashtags ---")
print(top_hashtags.to_string())

# Top 10 criadores por número de vídeos
top_creators = df_videos["author_name"].value_counts().head(10)
print("\n--- Top 10 Criadores (por nº de vídeos) ---")
print(top_creators.to_string())

# Gráficos
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) Histograma de likes (clipado no percentil 95)
likes_clip = df_videos["likes"].clip(upper=df_videos["likes"].quantile(0.95))
axes[0].hist(likes_clip, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribuição de Likes (p95)")
axes[0].set_xlabel("Likes")
axes[0].set_ylabel("Frequência")

# 2) Barras horizontais das top 10 hashtags
top10_tags = df_hashtags["hashtag"].value_counts().head(10)
axes[1].barh(top10_tags.index[::-1], top10_tags.values[::-1], color="coral")
axes[1].set_title("Top 10 Hashtags")
axes[1].set_xlabel("Contagem")

# 3) Histograma de comentários por vídeo
comments_per_video = df_comments.groupby("video_id").size()
axes[2].hist(comments_per_video, bins=30, color="mediumseagreen", edgecolor="white")
axes[2].set_title("Comentários por Vídeo")
axes[2].set_xlabel("Nº de Comentários")
axes[2].set_ylabel("Frequência")

plt.tight_layout()
plt.savefig("/content/eda_tiktok.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Gráfico salvo em /content/eda_tiktok.png")

In [ ]:
# ============================================================
# CÉLULA 7 — Rede 1: Bipartita Usuário × Hashtag (2-mode)
# ============================================================

import sqlite3
import networkx as nx
from networkx.algorithms import bipartite
from collections import defaultdict

DB_PATH = "/content/tiktok_sna.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Query: autor ↔ hashtag com peso
query = """
    SELECT v.author_id, vh.hashtag, COUNT(*) AS peso
    FROM videos v
    JOIN video_hashtags vh ON v.video_id = vh.video_id
    GROUP BY v.author_id, vh.hashtag
"""
rows = cur.execute(query).fetchall()
conn.close()

# Constrói grafo bipartido
G_bip = nx.Graph()

author_nodes = set()
hashtag_nodes = set()

for author_id, hashtag, peso in rows:
    a_node = f"u:{author_id}"
    h_node = f"#{hashtag}"

    if a_node not in author_nodes:
        G_bip.add_node(a_node, bipartite=0, node_type="author")
        author_nodes.add(a_node)

    if h_node not in hashtag_nodes:
        G_bip.add_node(h_node, bipartite=1, node_type="hashtag")
        hashtag_nodes.add(h_node)

    G_bip.add_edge(a_node, h_node, weight=peso)

# Projeções ponderadas
G_proj_user = bipartite.weighted_projected_graph(G_bip, author_nodes)
G_proj_tag = bipartite.weighted_projected_graph(G_bip, hashtag_nodes)

print("=" * 50)
print("REDE 1 — Bipartita: Usuário × Hashtag")
print("=" * 50)
print(f"Nós autores:    {len(author_nodes)}")
print(f"Nós hashtags:   {len(hashtag_nodes)}")
print(f"Arestas:        {G_bip.number_of_edges()}")
print(f"Densidade:      {nx.density(G_bip):.6f}")
print(f"\nProjeção Usuários: {G_proj_user.number_of_nodes()} nós, {G_proj_user.number_of_edges()} arestas")
print(f"Projeção Hashtags: {G_proj_tag.number_of_nodes()} nós, {G_proj_tag.number_of_edges()} arestas")

In [ ]:
# ============================================================
# CÉLULA 8 — Rede 2: Dígrafo Usuário × Usuário (Comentários)
# ============================================================

import sqlite3
import networkx as nx

DB_PATH = "/content/tiktok_sna.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Query: comentador → autor do vídeo (excluindo auto-comentários)
query = """
    SELECT c.commenter_id AS source,
           v.author_id   AS target,
           COUNT(*)      AS peso
    FROM comments c
    JOIN videos v ON c.video_id = v.video_id
    WHERE c.commenter_id != v.author_id
    GROUP BY c.commenter_id, v.author_id
"""
rows = cur.execute(query).fetchall()

# Mapas de nomes para atributos
names_map = {}
for row in cur.execute("SELECT author_id, author_name FROM videos").fetchall():
    names_map[row[0]] = row[1]
for row in cur.execute("SELECT commenter_id, commenter_name FROM comments").fetchall():
    if row[0] not in names_map:
        names_map[row[0]] = row[1]

conn.close()

# Constrói dígrafo
G_user = nx.DiGraph()

for source, target, peso in rows:
    if not G_user.has_node(source):
        G_user.add_node(source, name=names_map.get(source, source), node_type="user")
    if not G_user.has_node(target):
        G_user.add_node(target, name=names_map.get(target, target), node_type="user")
    G_user.add_edge(source, target, weight=peso)

print("=" * 50)
print("REDE 2 — Dígrafo: Usuário × Usuário (Comentários)")
print("=" * 50)
print(f"Nós:           {G_user.number_of_nodes()}")
print(f"Arestas:       {G_user.number_of_edges()}")
print(f"Densidade:     {nx.density(G_user):.6f}")
print(f"Reciprocidade: {nx.overall_reciprocity(G_user):.6f}")

In [ ]:
# ============================================================
# CÉLULA 9 — Rede 3: Co-ocorrência Hashtag × Hashtag
# ============================================================

import sqlite3
import pandas as pd
import networkx as nx
import itertools
from collections import defaultdict

DB_PATH = "/content/tiktok_sna.db"
conn = sqlite3.connect(DB_PATH)
df_hashtags = pd.read_sql("SELECT * FROM video_hashtags", conn)
conn.close()

# Agrupa hashtags por vídeo
tags_per_video = df_hashtags.groupby("video_id")["hashtag"].apply(list)

# Conta pares de co-ocorrência
pair_counts = defaultdict(int)
for tags in tags_per_video:
    for a, b in itertools.combinations(sorted(set(tags)), 2):
        pair_counts[(a, b)] += 1

# Constrói grafo de co-ocorrência
G_tag = nx.Graph()
for (a, b), cnt in pair_counts.items():
    G_tag.add_edge(f"#{a}", f"#{b}", weight=cnt)

# Maior componente conexa (GCC)
if G_tag.number_of_nodes() > 0:
    gcc_nodes = max(nx.connected_components(G_tag), key=len)
    G_tag_gcc = G_tag.subgraph(gcc_nodes).copy()
else:
    G_tag_gcc = G_tag.copy()

print("=" * 50)
print("REDE 3 — Co-ocorrência: Hashtag × Hashtag")
print("=" * 50)
print(f"Nós total:         {G_tag.number_of_nodes()}")
print(f"Arestas total:     {G_tag.number_of_edges()}")
print(f"Densidade:         {nx.density(G_tag):.6f}")
print(f"Tamanho da GCC:    {G_tag_gcc.number_of_nodes()} nós, {G_tag_gcc.number_of_edges()} arestas")

if nx.is_connected(G_tag_gcc) and G_tag_gcc.number_of_nodes() > 1:
    print(f"Diâmetro (GCC):    {nx.diameter(G_tag_gcc)}")
    print(f"Dist. média (GCC): {nx.average_shortest_path_length(G_tag_gcc):.4f}")
else:
    print("Diâmetro (GCC):    N/A (grafo desconectado ou trivial)")

In [ ]:
# ============================================================
# CÉLULA 10 — Métricas SNA + PageRank para as Três Redes
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import networkx as nx
import pandas as pd
import community as community_louvain


def metricas_nao_dirigido(G, nome, top_n=15):
    """Calcula métricas SNA para grafo não-dirigido."""
    print("\n" + "=" * 60)
    print(f"  {nome}")
    print("=" * 60)

    # Centralidades
    degree = dict(G.degree(weight="weight"))
    closeness = nx.closeness_centrality(G)
    betweenness = nx.betweenness_centrality(G, normalized=True, weight="weight")
    pagerank = nx.pagerank(G, alpha=0.85, weight="weight", max_iter=200)

    # Comunidades Louvain
    partition = community_louvain.best_partition(G, weight="weight", random_state=42)
    modularity = community_louvain.modularity(partition, G, weight="weight")
    n_coms = len(set(partition.values()))

    # Propriedades globais
    print(f"Nós:          {G.number_of_nodes()}")
    print(f"Arestas:      {G.number_of_edges()}")
    print(f"Densidade:    {nx.density(G):.6f}")
    print(f"Comunidades:  {n_coms}")
    print(f"Modularidade: {modularity:.4f}")

    # Diâmetro e distância média
    if nx.is_connected(G) and G.number_of_nodes() > 1:
        print(f"Diâmetro:     {nx.diameter(G)}")
        print(f"Dist. média:  {nx.average_shortest_path_length(G):.4f}")
    else:
        gcc_nodes = max(nx.connected_components(G), key=len)
        gcc = G.subgraph(gcc_nodes)
        if gcc.number_of_nodes() > 1:
            print(f"Diâmetro (GCC, {gcc.number_of_nodes()} nós): {nx.diameter(gcc)}")
            print(f"Dist. média (GCC): {nx.average_shortest_path_length(gcc):.4f}")

    # DataFrame de métricas
    df = pd.DataFrame({
        "no": list(G.nodes()),
        "degree": [degree[n] for n in G.nodes()],
        "closeness": [closeness[n] for n in G.nodes()],
        "betweenness": [betweenness[n] for n in G.nodes()],
        "pagerank": [pagerank[n] for n in G.nodes()],
        "comunidade": [partition[n] for n in G.nodes()],
    })

    # Rankings
    print(f"\n--- Top {top_n} por PageRank ---")
    print(df.nlargest(top_n, "pagerank")[["no", "pagerank", "comunidade"]].to_string(index=False))

    print(f"\n--- Top {top_n} por Betweenness ---")
    print(df.nlargest(top_n, "betweenness")[["no", "betweenness", "comunidade"]].to_string(index=False))

    print(f"\n--- Top {top_n} por Closeness ---")
    print(df.nlargest(top_n, "closeness")[["no", "closeness", "comunidade"]].to_string(index=False))

    return df, partition


def metricas_dirigido(G, nome, top_n=15):
    """Calcula métricas SNA para grafo dirigido."""
    print("\n" + "=" * 60)
    print(f"  {nome}")
    print("=" * 60)

    # Centralidades
    in_degree = dict(G.in_degree(weight="weight"))
    out_degree = dict(G.out_degree(weight="weight"))
    betweenness = nx.betweenness_centrality(G, normalized=True, weight="weight")
    closeness = nx.closeness_centrality(G)
    pagerank = nx.pagerank(G, alpha=0.85, weight="weight", max_iter=200)

    # Comunidades Louvain (na versão não-dirigida)
    G_und = G.to_undirected()
    partition = community_louvain.best_partition(G_und, weight="weight", random_state=42)
    modularity = community_louvain.modularity(partition, G_und, weight="weight")
    n_coms = len(set(partition.values()))

    # Propriedades globais
    print(f"Nós:           {G.number_of_nodes()}")
    print(f"Arestas:       {G.number_of_edges()}")
    print(f"Densidade:     {nx.density(G):.6f}")
    print(f"Reciprocidade: {nx.overall_reciprocity(G):.6f}")
    print(f"Comunidades:   {n_coms}")
    print(f"Modularidade:  {modularity:.4f}")

    # DataFrame de métricas
    df = pd.DataFrame({
        "no": list(G.nodes()),
        "in_degree": [in_degree[n] for n in G.nodes()],
        "out_degree": [out_degree[n] for n in G.nodes()],
        "betweenness": [betweenness[n] for n in G.nodes()],
        "closeness": [closeness[n] for n in G.nodes()],
        "pagerank": [pagerank[n] for n in G.nodes()],
        "comunidade": [partition[n] for n in G.nodes()],
    })

    # Rankings
    print(f"\n--- Top {top_n} por PageRank ---")
    print(df.nlargest(top_n, "pagerank")[["no", "pagerank", "comunidade"]].to_string(index=False))

    print(f"\n--- Top {top_n} por In-Degree ---")
    print(df.nlargest(top_n, "in_degree")[["no", "in_degree", "comunidade"]].to_string(index=False))

    print(f"\n--- Top {top_n} por Out-Degree ---")
    print(df.nlargest(top_n, "out_degree")[["no", "out_degree", "comunidade"]].to_string(index=False))

    print(f"\n--- Top {top_n} por Betweenness ---")
    print(df.nlargest(top_n, "betweenness")[["no", "betweenness", "comunidade"]].to_string(index=False))

    return df, partition


# === Execução nas três redes ===
df_bip_m,  part_bip  = metricas_nao_dirigido(G_bip,     "REDE 1 — Bipartita: Usuário × Hashtag")
df_user_m, part_user = metricas_dirigido(    G_user,    "REDE 2 — Dígrafo: Usuário × Usuário")
df_tag_m,  part_tag  = metricas_nao_dirigido(G_tag_gcc, "REDE 3 — Co-ocorrência: Hashtag × Hashtag (GCC)")

# Salva CSVs
df_bip_m.to_csv("/content/metricas_rede_bipartita.csv", index=False)
df_user_m.to_csv("/content/metricas_rede_usuarios.csv", index=False)
df_tag_m.to_csv("/content/metricas_rede_hashtags.csv", index=False)

print("\n" + "=" * 50)
print("CSVs salvos:")
for path, df in [("metricas_rede_bipartita.csv", df_bip_m),
                 ("metricas_rede_usuarios.csv", df_user_m),
                 ("metricas_rede_hashtags.csv", df_tag_m)]:
    print(f"  /content/{path} — colunas: {list(df.columns)}")

In [ ]:
# ============================================================
# CÉLULA 11 — Visualização Inline das Três Redes
# ============================================================

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from math import sqrt


def plota_rede(G, titulo, partition, df_metricas, id_col="no",
               layout_func=None, figsize=(14, 10), max_nos=120):
    """Plota rede com nós dimensionados por PageRank e coloridos por comunidade."""

    # Filtra top nós por PageRank se necessário
    if G.number_of_nodes() > max_nos:
        top_nodes = df_metricas.nlargest(max_nos, "pagerank")[id_col].tolist()
        G = G.subgraph([n for n in G.nodes() if n in top_nodes]).copy()
        # Atualiza partition para nós filtrados
        partition = {n: partition.get(n, 0) for n in G.nodes()}

    if G.number_of_nodes() == 0:
        print(f"⚠️ Rede '{titulo}' vazia, pulando visualização.")
        return

    n_nos = G.number_of_nodes()

    # Layout
    if layout_func is not None:
        pos = layout_func(G)
    else:
        k_val = 1.5 / sqrt(n_nos) if n_nos > 1 else 1.0
        pos = nx.spring_layout(G, seed=42, k=k_val)

    # PageRank para tamanho dos nós
    pr = nx.pagerank(G, alpha=0.85, weight="weight", max_iter=200)
    pr_max = max(pr.values()) if pr else 1
    node_sizes = [300 + (pr[n] / pr_max) * 2500 for n in G.nodes()]

    # Cores por comunidade
    coms = [partition.get(n, 0) for n in G.nodes()]
    n_coms = len(set(coms))
    cmap = cm.get_cmap("tab20", max(n_coms, 1))
    node_colors = [cmap(c) for c in coms]

    # Espessura das arestas
    weights = [G[u][v].get("weight", 1) for u, v in G.edges()]
    max_peso = max(weights) if weights else 1
    edge_widths = [0.3 + 3.0 * (w / max_peso) for w in weights]

    # Labels: top 20 por PageRank, truncados em 18 chars
    top20_pr = sorted(pr, key=pr.get, reverse=True)[:20]
    labels = {n: (n[:18] if len(n) <= 18 else n[:15] + "...") for n in top20_pr}

    # Desenha
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.set_title(titulo, fontsize=14, fontweight="bold")

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, width=edge_widths, edge_color="gray")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                          node_color=node_colors, alpha=0.85, edgecolors="white", linewidths=0.5)
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7, font_weight="bold")

    # Legenda de comunidades (máx 10)
    unique_coms = sorted(set(coms))[:10]
    legend_handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=cmap(c),
                   markersize=8, label=f"Comunidade {c}")
        for c in unique_coms
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=7, framealpha=0.8)

    ax.axis("off")
    plt.tight_layout()

    # Salva PNG
    fname = titulo.lower().replace(" ", "_").replace(":", "").replace("×", "x")
    fname = fname.replace("(", "").replace(")", "").replace("#", "").replace("-", "")
    fpath = f"/content/{fname}.png"
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"📊 Rede salva em {fpath}")


# === Plota as três redes ===
plota_rede(G_bip,     "Rede 1 - Bipartita: Usuario x Hashtag",
           part_bip,  df_bip_m)
plota_rede(G_user,    "Rede 2 - Digrafo: Usuario x Usuario",
           part_user, df_user_m)
plota_rede(G_tag_gcc, "Rede 3 - Co-ocorrencia: Hashtag x Hashtag",
           part_tag,  df_tag_m)

In [ ]:
# ============================================================
# CÉLULA 12 — Dashboard Comparativo das Métricas
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import os

# Configuração dos dados das três redes
redes = [
    ("Rede 1 - Bipartita", df_bip_m, part_bip, "pagerank", "betweenness"),
    ("Rede 2 - Dígrafo",   df_user_m, part_user, "pagerank", "betweenness"),
    ("Rede 3 - Hashtags",  df_tag_m, part_tag, "pagerank", "betweenness"),
]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle("Dashboard SNA — #inteligenciaartificial (TikTok)",
             fontsize=16, fontweight="bold", y=0.98)

for row_idx, (rede_nome, df, partition, pr_col, bt_col) in enumerate(redes):

    # Coluna 0: Top 10 por PageRank (barras horizontais)
    ax0 = axes[row_idx, 0]
    top10_pr = df.nlargest(10, pr_col)
    labels_pr = [str(n)[:20] for n in top10_pr["no"]]
    ax0.barh(range(len(labels_pr)), top10_pr[pr_col].values, color="steelblue")
    ax0.set_yticks(range(len(labels_pr)))
    ax0.set_yticklabels(labels_pr, fontsize=7)
    ax0.invert_yaxis()
    ax0.set_title(f"{rede_nome}\nTop 10 PageRank", fontsize=9, fontweight="bold")
    ax0.set_xlabel("PageRank", fontsize=8)

    # Coluna 1: Top 10 por Betweenness (barras horizontais)
    ax1 = axes[row_idx, 1]
    top10_bt = df.nlargest(10, bt_col)
    labels_bt = [str(n)[:20] for n in top10_bt["no"]]
    ax1.barh(range(len(labels_bt)), top10_bt[bt_col].values, color="coral")
    ax1.set_yticks(range(len(labels_bt)))
    ax1.set_yticklabels(labels_bt, fontsize=7)
    ax1.invert_yaxis()
    ax1.set_title(f"{rede_nome}\nTop 10 Betweenness", fontsize=9, fontweight="bold")
    ax1.set_xlabel("Betweenness", fontsize=8)

    # Coluna 2: Scatter PageRank × Betweenness colorido por comunidade
    ax2 = axes[row_idx, 2]
    coms = df["comunidade"].values
    n_coms = len(set(coms))
    cmap = cm.get_cmap("tab20", max(n_coms, 1))
    colors = [cmap(c) for c in coms]

    ax2.scatter(df[pr_col], df[bt_col], c=colors, alpha=0.7, s=30, edgecolors="white", linewidths=0.3)
    ax2.set_xlabel("PageRank", fontsize=8)
    ax2.set_ylabel("Betweenness", fontsize=8)
    ax2.set_title(f"{rede_nome}\nPageRank × Betweenness", fontsize=9, fontweight="bold")

    # Anotação dos top 5 por PageRank
    top5 = df.nlargest(5, pr_col)
    for _, r in top5.iterrows():
        label = str(r["no"])[:15]
        ax2.annotate(label, (r[pr_col], r[bt_col]), fontsize=6, alpha=0.8)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("/content/dashboard_sna.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Dashboard salvo em /content/dashboard_sna.png")

# Lista todos os arquivos gerados
print("\n" + "=" * 50)
print("ARQUIVOS GERADOS EM /content/")
print("=" * 50)
for f in sorted(os.listdir("/content/")):
    if f.endswith((".png", ".csv", ".db")):
        size_kb = os.path.getsize(f"/content/{f}") / 1024
        print(f"  {f:45s} {size_kb:8.1f} KB")